# 01 · Data quality, cleaning, and SQL modeling

Build an order-level analysis that preserves financial totals and handles missing dates and repeat reviews consistently. The previous notebook established why the raw child tables need pre-aggregation.

[Previous](00_source_and_relational_schema.ipynb) · [Notebook guide](../README.md#notebook-guide) · [Next](02_delivery_and_satisfaction.ipynb)

In [1]:
import json
import sqlite3
from contextlib import closing
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src/olist_analytics").is_dir(), (
    "Open from the project root or notebooks directory."
)
REPORTS = ROOT / "reports"
TABLES = REPORTS / "tables"


DB = ROOT / "data/processed/olist.sqlite3"
DB_URI = DB.as_uri() + "?mode=ro"

## Which missing fields affect the analysis?

The pipeline removes exact duplicates in derived data, checks keys and preserves null source values. Use the recorded field-level counts to distinguish relevant missing dates from optional review text.

In [2]:
quality = json.loads((REPORTS / "data_quality.json").read_text())
missing = pd.DataFrame(
    (name, field, count, count / info["raw_rows"])
    for name, info in quality.items()
    if isinstance(info, dict) and "missing_raw" in info
    for field, count in info["missing_raw"].items()
    if count
)
missing.columns = ["table", "field", "missing_rows", "missing_share"]
display(missing.nlargest(10, "missing_share"))
assert quality["passed"]

,table,field,missing_rows,missing_share
0,order_reviews,review_comment_title,87656,0.883415
1,order_reviews,review_comment_message,58247,0.587025
4,orders,order_delivered_customer_date,2965,0.029817
5,products,product_category_name,610,0.018512
6,products,product_name_lenght,610,0.018512
7,products,product_description_lenght,610,0.018512
8,products,product_photos_qty,610,0.018512
3,orders,order_delivered_carrier_date,1783,0.017930
2,orders,order_approved_at,160,0.001609
9,products,product_weight_g,2,0.000061


Optional review comments account for substantial missingness. Scores and their timestamps govern the review analysis. Delivery eligibility depends on the three date fields and delivery status.

## How does the model preserve one row per order?

The CTEs aggregate payments and items before joining. ROW_NUMBER selects one review with deterministic tie-breaking. CASE WHEN defines an eligible delivery, compares calendar dates and assigns null lateness outside the cohort. Read the complete SQL and check the stored view.

In [3]:
print((ROOT / "sql/02_order_fact.sql").read_text())
with closing(sqlite3.connect(DB_URI, uri=True)) as connection:
    counts = pd.read_sql_query(
        """
        SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS orders
        FROM order_fact
    """,
        connection,
    )
assert counts.at[0, "rows"] == counts.at[0, "orders"]
display(counts)
model = json.loads((REPORTS / "model_validation.json").read_text())
assert model["passed"]
display(pd.DataFrame(model["money_reconciliation"]).T)

-- Aggregate each one-to-many relationship BEFORE joining. One row per order.
CREATE VIEW order_fact AS
WITH item_totals AS (
 SELECT order_id, COUNT(*) AS item_count, COUNT(DISTINCT seller_id) AS seller_count,
        SUM(price) AS merchandise_value_brl, SUM(freight_value) AS freight_value_brl
 FROM order_items GROUP BY order_id
), payment_totals AS (
 SELECT order_id, COUNT(*) AS payment_record_count, SUM(payment_value) AS payment_value_brl
 FROM order_payments GROUP BY order_id
), ranked_reviews AS (
 SELECT *, ROW_NUMBER() OVER (
   PARTITION BY order_id
   ORDER BY review_answer_timestamp DESC, review_creation_date DESC, review_id DESC,
            review_score DESC, COALESCE(review_comment_title, '') DESC,
            COALESCE(review_comment_message, '') DESC
 ) AS review_rank, COUNT(*) OVER (PARTITION BY order_id) AS review_count
 FROM order_reviews
), base AS (
 SELECT o.*, c.customer_unique_id, c.customer_state, c.customer_city,
        STRFTIME('%Y-%m', o.order_purchase_times

,rows,orders
0,99441,99441


,source_total,fact_total,absolute_difference
merchandise_value_brl,13591643.70,13591643.70,0.000000e+00
freight_value_brl,2251909.54,2251909.54,0.000000e+00
payment_value_brl,16008872.12,16008872.12,1.862645e-09


The model retains 99,441 unique orders. Product price, freight and payment totals each reconcile within a cent. Treat the promised date as inclusive through the calendar day.

## Who enters the delivery denominator?

In [4]:
exclusions_sql = (ROOT / "sql/08_audit.sql").read_text()
print(exclusions_sql)
with closing(sqlite3.connect(DB_URI, uri=True)) as connection:
    exclusions = pd.read_sql_query(exclusions_sql, connection)
display(exclusions)

-- Exclusion reasons are hierarchical, so row counts sum to the full orders table.
SELECT CASE
 WHEN order_status <> 'delivered' THEN 'Not delivered status'
 WHEN order_delivered_customer_date IS NULL THEN 'Delivered status; missing actual delivery'
 WHEN order_estimated_delivery_date IS NULL THEN 'Missing estimated delivery'
 WHEN JULIANDAY(order_delivered_customer_date) < JULIANDAY(order_purchase_timestamp)
   THEN 'Actual delivery before purchase'
 WHEN DATE(order_estimated_delivery_date) < DATE(order_purchase_timestamp)
   THEN 'Estimated delivery before purchase date'
 ELSE 'Eligible delivered' END AS cohort,
 COUNT(*) AS orders
FROM order_fact GROUP BY cohort ORDER BY orders DESC;



,cohort,orders
0,Eligible delivered,96470
1,Not delivered status,2963
2,Delivered status; missing actual delivery,8


The eligibility audit retains 96,470 orders for delivery comparisons. Cancellations and other undelivered statuses remain visible in the wider order population. Next, compare the verified KPI and investigate where late deliveries are concentrated.

[Previous](00_source_and_relational_schema.ipynb) · [Notebook guide](../README.md#notebook-guide) · [Next](02_delivery_and_satisfaction.ipynb)